# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an mlcroissant.DatasetMetadata object

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset version: {metadata.version}")
print(f"Dataset license: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their @id, and their fields
if not metadata.recordSets:
    print("No record sets defined in the dataset metadata!")
else:
    for record_set in metadata.recordSets:
        print(f"Record Set: {record_set.name} (id: {record_set.id})")
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field.name} (id: {field.id}, type: {field.dataType})")
        print()
    # For exploration in further steps, collect the list of record set ids
    record_set_ids = [rs.id for rs in metadata.recordSets]
    if record_set_ids:
        print("Available record set ids:", record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract records for each record set dynamically by their @id
dataframes = {}
if not metadata.recordSets:
    print("No record sets present for extraction.")
else:
    # We'll load all record sets found.
    for record_set in metadata.recordSets:
        records = list(dataset.records(record_set=record_set.id))
        df = pd.DataFrame(records)
        dataframes[record_set.id] = df
        print(f"Loaded {len(df)} records for record set '{record_set.name}' (id: {record_set.id})")
    # Pick the first record set for example exploration
    main_record_set_id = metadata.recordSets[0].id
    print("\nColumns available in first record set (using only @id as reference):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Filter and normalize a numeric field

if dataframes:
    df = dataframes[main_record_set_id]
    # Identify a suitable numeric field by type using record set metadata
    
    main_record_set = None
    for rs in metadata.recordSets:
        if rs.id == main_record_set_id:
            main_record_set = rs
            break
    numeric_field_id = None
    group_field_id = None
    if main_record_set:
        for field in main_record_set.fields:
            if field.dataType in ['Float', 'Integer', 'Number'] and numeric_field_id is None:
                numeric_field_id = field.id
            if field.dataType == 'Text' and group_field_id is None:
                group_field_id = field.id
    # If no numeric field, the rest will not run
    if numeric_field_id and numeric_field_id in df.columns:
        # Remove missing/NaN
        sdf = df[df[numeric_field_id].notnull()]
        # Sometimes Croissant/CSV may load numbers as strings, so cast
        sdf[numeric_field_id] = pd.to_numeric(sdf[numeric_field_id], errors='coerce')
        # Filter (example: values > mean)
        mean_val = sdf[numeric_field_id].mean()
        filtered_df = sdf[sdf[numeric_field_id] > mean_val]
        print(f"Filtered records with {numeric_field_id} > mean ({mean_val:.2f}) :")
        print(filtered_df.head())

        # Normalization (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If a grouping field exists
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No group field available for grouping.")
    else:
        print("No numeric field available in the main record set.")
else:
    print("No DataFrames loaded for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting the distribution of the selected numeric field (if available)
if dataframes and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print('No suitable numeric field available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
* Loaded and summarized the [FAIR<sup>2</sup> dataset package: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
* Inspected the available record sets and fields using their Croissant `@id` values
* Loaded the main protein record set into a DataFrame and demonstrated field extraction using only `@id`
* Performed basic filtering, normalization, and grouping on a numeric field
* Visualized the field distribution

You can further explore specific proteins, abundance levels, or molecular weights, or extend the notebook to compare groups and investigate data quality or outliers using the unified structure provided by the Croissant standard.